# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in successfully")

Logged in successfully


In [2]:
from datasets import load_dataset

dim_clients = load_dataset("FlyRank/internship-warehouse", "dim_clients", split="train").to_pandas()
dim_content = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train").to_pandas()
fact_query  = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", split="train").to_pandas()

print(dim_clients.shape, dim_content.shape, fact_query.shape)

(104, 9) (519606, 26) (2414248, 21)


In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
march_files = [f for f in files if "fact_content_daily_performance/month=2026-03" in f]
print(march_files)

['fact_content_daily_performance/month=2026-03/data_0.parquet']


In [4]:
from huggingface_hub import hf_hub_download
import pandas as pd

local_paths = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", filename=f, repo_type="dataset")
    for f in march_files
]
sample_month = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
print(sample_month.shape)
print(sample_month.columns.tolist())

(9841378, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row = one client's one content item's organic performance on one report_date.
Time window: report_date within March 2026 (month=2026-03) — a mid-panel month, not the sealed final month (June 2026).
dim_clients and dim_content are current snapshots with no time window (context only).

In [5]:
# Grain check
dupes = sample_month.groupby(['client_hash_id','content_hash_id','report_date']).size()
print("Rows with count > 1:", (dupes > 1).sum())  # expect 0

# Row count and date span
print("Total rows:", len(sample_month))
print(sample_month.groupby('client_hash_id')['report_date'].agg(['min','max']).head())

Rows with count > 1: 0
Total rows: 9841378
                                min         max
client_hash_id                                 
client_0797ff3a1fc9a6a5  2026-03-01  2026-03-31
client_08a6a72ff48e62c0  2026-03-01  2026-03-31
client_08d2847f24cf89c1  2026-03-01  2026-03-31
client_0e1acc6cd57b0eba  2026-03-01  2026-03-31
client_0fa64a184f18a4a0  2026-03-01  2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


- Feature: gsc_clicks_7d_avg, gsc_impressions_7d_avg, gsc_ctr, gsc_avg_position, client_has_gsc_flag
- Label: next-month gsc_clicks growth (computed from future rows — never used as an input feature)
- Context: client_hash_id, content_hash_id, report_date (used for grouping/joining/splitting, not modeling)
- Excluded: fact_content_query_90d — different grain (client + content + query over a rolling 90-day window), out of scope for this monthly-grain lane; would require separate join logic

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
missing_pct = sample_month.isnull().mean()
print(missing_pct[missing_pct > 0])

ga4_data_available          0.306740
gsc_avg_position            0.633074
ga4_pageviews               0.306740
ga4_sessions                0.306740
ga4_users                   0.306740
ga4_engaged_sessions        0.306740
ga4_total_engagement_sec    0.306740
sessions_organic            0.306740
sessions_direct             0.306740
sessions_referral           0.306740
sessions_social             0.306740
sessions_paid               0.306740
sessions_ai                 0.306740
ai_chatgpt                  0.306740
ai_perplexity               0.306740
ai_gemini                   0.306740
ai_copilot                  0.306740
ai_claude                   0.306740
ai_meta                     0.306740
ai_other                    0.306740
scroll_events               0.306740
dtype: float64


In [7]:
gsc_available = sample_month[sample_month['gsc_data_available'] == True]
ga4_available = sample_month[sample_month['ga4_data_available'] == True]
print("GSC available:", len(gsc_available), "of", len(sample_month))
print("GA4 available:", len(ga4_available), "of", len(sample_month))

GSC available: 3611061 of 9841378
GA4 available: 413966 of 9841378


In [8]:
features = sample_month[sample_month['gsc_data_available'] == True].copy()

features['gsc_clicks_7d_avg'] = features.groupby('content_hash_id')['gsc_clicks'].transform(lambda x: x.rolling(7, min_periods=1).mean())
features['gsc_impressions_7d_avg'] = features.groupby('content_hash_id')['gsc_impressions'].transform(lambda x: x.rolling(7, min_periods=1).mean())
features['gsc_ctr'] = features['gsc_clicks'] / features['gsc_impressions'].replace(0, 1)
features['client_has_gsc_flag'] = features['client_has_gsc'].astype(int)
# gsc_avg_position is already a direct column — no computation needed

print(features[['gsc_clicks_7d_avg','gsc_impressions_7d_avg','gsc_ctr','gsc_avg_position','client_has_gsc_flag']].describe())

       gsc_clicks_7d_avg  gsc_impressions_7d_avg       gsc_ctr  \
count       3.611061e+06            3.611061e+06  3.611061e+06   
mean        2.295190e-01            7.686100e+01  3.080748e-03   
std         1.115234e+00            2.246838e+02  3.009151e-02   
min         0.000000e+00            1.000000e+00  0.000000e+00   
25%         0.000000e+00            4.571429e+00  0.000000e+00   
50%         0.000000e+00            1.660000e+01  0.000000e+00   
75%         1.428571e-01            6.400000e+01  0.000000e+00   
max         2.311429e+02            3.505914e+04  1.000000e+00   

       gsc_avg_position  client_has_gsc_flag  
count      3.611061e+06            3611061.0  
mean       1.582665e+01                  1.0  
std        1.985603e+01                  0.0  
min        0.000000e+00                  1.0  
25%        3.742120e+00                  1.0  
50%        7.500000e+00                  1.0  
75%        2.020000e+01                  1.0  
max        4.980000e+02      

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



- Unbalanced history: gsc_data_start / ga4_data_start vary per client (see dim_clients) — this is an unbalanced panel, not a random sample of history.
- GSC availability: only 36.69% of rows (3,611,061 of 9,841,378) have gsc_data_available=True. Where it's False, gsc_avg_position and related GSC fields are not real measurements — 63.31% of rows are missing gsc_avg_position for this reason.
- GA4 availability: only 4.21% of rows (413,966 of 9,841,378) have ga4_data_available=True. The remaining ~95.79% have zero-filled GA4 columns (ga4_pageviews, ga4_sessions, ga4_users, session-source breakdowns, AI-referral columns, scroll_events, etc.) that mean "no data," not real zero-engagement — using them as true zeros would badly understate engagement for most of the panel.
- Window overlaps: fact_content_query_90d uses a rolling 90-day window, which overlaps across calendar months — its numbers are not additive with this monthly-grain slice.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.